### Set up Project Variables and Import Database

setting up neo4j locally - 

1. docker pull neo4j

2. docker stop neo4j
docker rm neo4j
docker run -d \
  --name neo4j \
  -p 7474:7474 -p 7687:7687 \
  -e NEO4J_AUTH=neo4j/`your_password` \
  -e NEO4J_PLUGINS='["apoc", "gds"]' \
  -e "NEO4J_dbms_security_procedures_unrestricted=apoc.*,gds.*" \
  neo4j




In [1]:
# from langchain_community.graphs import Neo4jGraph
from langchain_neo4j import Neo4jGraph
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langchain_nvidia_ai_endpoints import ChatNVIDIA
import os
from dotenv import load_dotenv


load_dotenv() # Load environment variables from .env file


NEO4J_URI = os.getenv("NEO4J_LOCAL_URI")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
NEO4J_USERNAME=os.getenv("NEO4J_LOCAL_USERNAME")
NEO4J_PASSWORD=os.getenv("NEO4J_LOCAL_PASSWORD")
os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USERNAME"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD

# graph = Neo4jGraph()
graph = Neo4jGraph(
    url=NEO4J_URI, 
    username=NEO4J_USERNAME, 
    password=NEO4J_PASSWORD
)

# Test the connection by printing the schema
graph.refresh_schema()
print(graph.schema)
llm=ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

Node properties:
Individual {name: STRING}
Relationship properties:

The relationships:
(:Individual)-[:FRIEND_OF]->(:Individual)


In [2]:
from utilities import extract_pdf_chunks, save_chunks_to_file

pdf_path="data/pdf/Commercial Exchange CHIP PT Minutes Sept 2024 eVote.pdf"

chunks = extract_pdf_chunks(pdf_path)
save_chunks_to_file(chunks, "data/text/drugs_context.txt")

In [3]:
content = ""
# Open the file in read mode
with open('data/text/drugs_context.txt', 'r') as file:
    # Read the contents of the file
    content = file.read()

In [7]:
content[:500]

'Page 1:\nPOLICIES AND PROCEDURE\nMANUAL\nP&T Committee Meeting Minutes\nCommercial, Exchange, CHIP\nSeptember 17th, 2024\nPresent (via Teams): Absent:\nBret Yarczower, MD, MBA – Chair Alyssa Cilia, RPh\nAmir Antonious, Pharm.D. Keri Donaldson, MD, MSCE\nEmily Bednarz, Pharm.D. Michael Evans, RPh\nKristen Bender, Pharm.D. Nichole Hossler, MD\nJeremy Bennett, MD Jonas Pearson, RPh\nKim Castelnovo, RPh Angela Scarantino\nKimberly Clark, Pharm.D. William Seavey, Pharm.D.\nBhargavi Degapudi, MD Michael Shepherd, M'

In [4]:
import re
parts = re.split(r'Page (\d+):', content)

parts[:5]

['',
 '1',
 '\nPOLICIES AND PROCEDURE\nMANUAL\nP&T Committee Meeting Minutes\nCommercial, Exchange, CHIP\nSeptember 17th, 2024\nPresent (via Teams): Absent:\nBret Yarczower, MD, MBA – Chair Alyssa Cilia, RPh\nAmir Antonious, Pharm.D. Keri Donaldson, MD, MSCE\nEmily Bednarz, Pharm.D. Michael Evans, RPh\nKristen Bender, Pharm.D. Nichole Hossler, MD\nJeremy Bennett, MD Jonas Pearson, RPh\nKim Castelnovo, RPh Angela Scarantino\nKimberly Clark, Pharm.D. William Seavey, Pharm.D.\nBhargavi Degapudi, MD Michael Shepherd, MD\nMichae\n\n',
 '2',
 '\nl Dubartell, MD\nKelly Faust, Pharm.D.\nTricia Heitzman, Pharm.D.\nJason Howay, Pharm.D.\nKeith Hunsicker, Pharm.D.\nKelli Hunsicker, Pharm.D.\nDerek Hunt, Pharm.D.\nEmily Jacobson, Pharm.D.\nDennis Janozczyk, Pharm.D.\nAlexandra Kempf-Malys, MSW, BSc\nKerry Ann Kilkenny, MD\nPhilip Krebs, R.EEG T\nBriana LeBeau, Pharm.D.\nTed Marines, Pharm.D.\nLisa Mazonkey, RPh\nTyreese McCrea, Pharm.D.\nPerry Meadows, MD\nJamie Miller, RPh\nMark Mowery, Pharm.D.\

In [5]:
content_list = []
for i in range(1, len(parts), 2):
    page_number = parts[i]
    page_content = parts[i+1].strip()
    content_list.append({
        'text': page_content,
        'page_number': page_number
    })

In [6]:
from langchain_core.documents import Document
from langchain_experimental.graph_transformers import LLMGraphTransformer

llm_transformer = LLMGraphTransformer(llm=llm)
langchain_docs = [
    Document(page_content=item['text'], metadata={"page_number": item['page_number']}) 
    for item in content_list
]
print(f"Total pages processed: {len(content_list)}")


Total pages processed: 302


In [8]:
graph_documents = []
for doc in langchain_docs:
    # This sends only ONE page at a time to Groq
    graph_doc = llm_transformer.convert_to_graph_documents([doc])
    graph_documents.extend(graph_doc)
print(f"Nodes:{graph_documents[0].nodes}")
print(f"Relationships:{graph_documents[0].relationships}")

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kft4wgv5ef6azxynxwngzfj9` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 100000, Requested 1239. Please try again in 17m50.496s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [7]:
import time
import re
from groq import RateLimitError # Make sure to import the specific error

def process_with_retry(llm_transformer, docs):
    graph_documents = []
    
    for i, doc in enumerate(docs):
        success = False
        while not success:
            try:
                print(f"Processing Page {doc.metadata['page_number']}...")
                # Attempt the extraction
                graph_doc = llm_transformer.convert_to_graph_documents([doc])
                graph_documents.extend(graph_doc)
                success = True
                # Optional: Small sleep to avoid hitting the 12,000 TPM limit
                time.sleep(2) 
                
            except RateLimitError as e:
                # Extract the wait time from the error message if possible
                # Otherwise, default to 60 seconds
                error_msg = str(e)
                wait_time = 60 # Default
                
                # Regex to find "try again in XmYs" or "Xs"
                match = re.search(r'try again in (\d+m)?([\d\.]+)s', error_msg)
                if match:
                    minutes = match.group(1)
                    seconds = float(match.group(2))
                    wait_time = (int(minutes[:-1]) * 60 if minutes else 0) + seconds + 2
                
                print(f"Rate limit hit. Sleeping for {wait_time} seconds...")
                time.sleep(wait_time)
                
            except Exception as e:
                print(f"An unexpected error occurred: {e}")
                break # Exit if it's not a rate limit issue
                
    return graph_documents

# Usage
# langchain_docs is the list we created in the previous step
final_graph_docs = process_with_retry(llm_transformer, langchain_docs)


Processing Page 1...
Processing Page 2...
Processing Page 3...
Processing Page 4...
Processing Page 5...
Processing Page 6...
Processing Page 7...
Processing Page 8...
Processing Page 9...
Processing Page 10...
Processing Page 11...
Processing Page 12...
Processing Page 13...
Processing Page 14...
Processing Page 15...
Processing Page 16...
Processing Page 17...
Processing Page 18...
Processing Page 19...
Processing Page 20...
Processing Page 21...
Processing Page 22...
Processing Page 23...
Processing Page 24...
Processing Page 25...
Processing Page 26...
Processing Page 27...
Processing Page 28...
Processing Page 29...
Processing Page 30...
Processing Page 31...
Processing Page 32...
Processing Page 33...
Processing Page 34...
Processing Page 35...
Processing Page 36...
Processing Page 37...
Processing Page 38...
Processing Page 39...
Processing Page 40...
Processing Page 41...
Processing Page 42...
An unexpected error occurred: Error code: 400 - {'error': {'message': "Failed to call

KeyboardInterrupt: 